In [1]:
# importing required classes
from pypdf import PdfReader

# creating a pdf reader object
reader = PdfReader('Maxis_HSBA_Funtional_Specification_Document_v1.3 1_Signed_off.pdf')

# printing number of pages in pdf file
print("Pages = ",len(reader.pages))

## Create a new text file and open it in write mode
with open("Maxis_HSBA_Funtional_Specification_Document_copy.txt", "w") as f:
    # creating a page object
    for i in range(18,358):
        page = reader.pages[i]
        text = page.extract_text()
        updated_text = text.replace(f"Functional  Specification  Document  Subex Limited  \nSubex Confidential Proprietary  {i-4}","")
          ## Write to the text file
        f.write(updated_text)

# extracting text from page
print(updated_text)

Pages =  362
 
                           
 
 
 
Confidential  Output  Top 10 Rate planamount wise trend  
Chart2  Billing -Enterprise  
Filter  Rate Plan  
Month/Day/Year  
Control point  RT_blng_items  
Metrics  Top 10 Rate planamount wise trend and summary table for all rate plans  
Output  Top 10 Rate planamount wise trend  
 
 


In [4]:
pip install -U sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import time

import numpy as np
import pandas as pd
import requests
import redis
from redis.commands.search.field import (
    NumericField,
    TagField,
    TextField,
    VectorField,
)
from redis.commands.search.indexDefinition import IndexDefinition, IndexType
from redis.commands.search.query import Query
from sentence_transformers import SentenceTransformer

/opt/conda/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


In [11]:
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import SimpleDirectoryReader

In [12]:
# load documents
# documents = SimpleDirectoryReader("./sample_data/").load_data()
documents = SimpleDirectoryReader("./maxis_data/").load_data()
# print(documents)
print("Document ID:", documents[0].doc_id)

Document ID: 4b58faa0-60d7-46ae-becd-90e4554591ba


In [13]:
base_splitter = SentenceSplitter(chunk_size=400, chunk_overlap=40)

nodes2 = base_splitter.get_nodes_from_documents(documents)

In [14]:
len(nodes2)

466

In [15]:
nodes2[0]

TextNode(id_='16f8579f-e541-4a8f-9153-df97411e5248', embedding=None, metadata={'file_path': '/home/jovyan/work/lakshman/maxis_data/Maxis_HSBA_Funtional_Specification_Document.txt', 'file_name': 'Maxis_HSBA_Funtional_Specification_Document.txt', 'file_type': 'text/plain', 'file_size': 443619, 'creation_date': '2024-10-07', 'last_modified_date': '2024-10-07'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='4b58faa0-60d7-46ae-becd-90e4554591ba', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'file_path': '/home/jovyan/work/lakshman/maxis_data/Maxis_HSBA_Funtional_Specification_Document.txt', 'file_name': 'Maxis_HSBA_Funtional_Specification_Document.txt', 'file_type': 'text/plain', 'file_size': 443619, 'creat

In [16]:
nodes2[0].get_content()

'Confidential  9.1 Usage_ALL_Roaming_VoiceSMS_MSCvsTAPOUT – ConEnt\nAudit Name  Usage_ALL_Roaming_VoiceSMS_MSCvsTAPOUT – ConEnt\n\nUsage_ALL_Roaming_VoiceSMS_MSCvsTAPOUT\nBRD Reference No  3.1.1 .1\nObjective  To identify discrepancy of count and sum of duration between Roaming\nVoiceSMS_MSCvsTAPOUT\nSources  • ericsson_msc_roaming\n• tapout\nFrequency of Audit  Daily\nMeasure Name  DM1 – First Level Recon Between MSC and TAPOUT(Inroamers)\nDescription  This measure gives one to one mapping of records from both the sources\nDM Source -1  ericsson_msc_roaming\nFilter Conditions  is_maxis_subscriber = N\nand call_duration <> 0\nand b_number <> 994\nand call_transaction_type<> transit -mt\nand imsi notlike 50218%\nand imsi notlike 502153%\nDM Source -2 tapout\nFilter Conditions  recordtype_cdr <> GPRS\nand duplicate_fl = N\nMatch Keys  ericsson_msc_roaming  tapout\nimsi\nstart_dttm\ncall_duration\nrecipient_plmn_id  Imsi\ncall_event_start_time\nchargeable_units\nrecipient\nColumn Selectio

In [17]:
%%time
# Get embeddings for all the nodes
documents_with_embeddings = []
for i in range(0,len(nodes2)):
    # print(i)
# Define the URL and the data
    url = 'http://localhost:11434/api/embeddings'
    data = {
      "model": "jmorgan/gte-small:latest", 
      "prompt": nodes2[i].get_content()
    }
    
    # Send the POST request
    response = requests.post(url, json=data)
    
    
    # Convert response to JSON (dict)
    response_data = response.json()
    response_map = {}
    response_map['content'] = nodes2[i].get_content()
    response_map['embedded_content'] = response_data['embedding']
    documents_with_embeddings.append(response_map)

# Print or manipulate the data
print(len(documents_with_embeddings))


466
CPU times: user 863 ms, sys: 92.4 ms, total: 955 ms
Wall time: 2min 53s


In [18]:
documents_with_embeddings[0]

{'content': 'Confidential  9.1 Usage_ALL_Roaming_VoiceSMS_MSCvsTAPOUT – ConEnt\nAudit Name  Usage_ALL_Roaming_VoiceSMS_MSCvsTAPOUT – ConEnt\n\nUsage_ALL_Roaming_VoiceSMS_MSCvsTAPOUT\nBRD Reference No  3.1.1 .1\nObjective  To identify discrepancy of count and sum of duration between Roaming\nVoiceSMS_MSCvsTAPOUT\nSources  • ericsson_msc_roaming\n• tapout\nFrequency of Audit  Daily\nMeasure Name  DM1 – First Level Recon Between MSC and TAPOUT(Inroamers)\nDescription  This measure gives one to one mapping of records from both the sources\nDM Source -1  ericsson_msc_roaming\nFilter Conditions  is_maxis_subscriber = N\nand call_duration <> 0\nand b_number <> 994\nand call_transaction_type<> transit -mt\nand imsi notlike 50218%\nand imsi notlike 502153%\nDM Source -2 tapout\nFilter Conditions  recordtype_cdr <> GPRS\nand duplicate_fl = N\nMatch Keys  ericsson_msc_roaming  tapout\nimsi\nstart_dttm\ncall_duration\nrecipient_plmn_id  Imsi\ncall_event_start_time\nchargeable_units\nrecipient\nCol

In [30]:
with open('maxis_documents_embeddings.json','w') as f :
    json.dump(documents_with_embeddings,f)

In [31]:
# Connect to Redis
client = redis.Redis(host='127.0.0.1', port=6379, socket_timeout=10)

In [32]:
client.ping()

True

In [33]:
pipeline = client.pipeline()
for i, items in enumerate(documents_with_embeddings, start=1):
    redis_key = f"maxis:{i}"
    pipeline.json().set(redis_key, "$", {"content": i, "embedded_content": items})
res = pipeline.execute()

In [45]:
# Define the index schema and create the index
VECTOR_DIMENSION = 384
schema = (
    TextField("$.content", as_name="content"),
    VectorField(
        "$.embedded_content", 
        "FLAT", 
        {
        "TYPE": "FLOAT32",
        "DIM": VECTOR_DIMENSION,
        "DISTANCE_METRIC": "IP"
    }, 
        as_name="vector")
)

definition = IndexDefinition(prefix=["maxis:"], index_type=IndexType.JSON)
# Create the index
res = client.ft("idx:maxis_documents").create_index(fields=schema, definition=definition)
res


b'OK'

In [46]:
info = client.ft("idx:maxis_documents").info()
# num_docs = info["num_docs"]
print(info)
print("num_docs", info["num_docs"])
indexing_failures = info["hash_indexing_failures"]
# print(f"{num_docs} documents indexed with {indexing_failures} failures")
# >>> 11 documents indexed with 0 failures

{'index_name': 'idx:maxis_documents', 'index_options': [], 'index_definition': [b'key_type', b'JSON', b'prefixes', [b'maxis:'], b'default_score', b'1'], 'attributes': [[b'identifier', b'$.content', b'attribute', b'content', b'type', b'TEXT', b'WEIGHT', b'1'], [b'identifier', b'$.embedded_content', b'attribute', b'vector', b'type', b'VECTOR', b'algorithm', b'FLAT', b'data_type', b'FLOAT32', b'dim', 384, b'distance_metric', b'IP']], 'num_docs': '0', 'max_doc_id': '0', 'num_terms': '0', 'num_records': '0', 'inverted_sz_mb': '0', 'vector_index_sz_mb': '0.00818634033203125', 'total_inverted_index_blocks': '0', 'offset_vectors_sz_mb': '0', 'doc_table_size_mb': '0', 'sortable_values_size_mb': '0', 'key_table_size_mb': '0', 'tag_overhead_sz_mb': '0', 'text_overhead_sz_mb': '0', 'total_index_memory_sz_mb': '0', 'geoshapes_sz_mb': '0', 'records_per_doc_avg': 'nan', 'bytes_per_record_avg': 'nan', 'offsets_per_term_avg': 'nan', 'offset_bits_per_record_avg': 'nan', 'hash_indexing_failures': '0', 't

In [49]:
%%time
# Query embeddings
encoded_queries = [[-0.08354659378528595,
  -0.16112320125102997,
  0.34088003635406494,
  -0.32282453775405884,
  -0.12185712158679962,
  0.046853527426719666,
  0.2821461260318756,
  0.310691237449646,
  0.31815972924232483,
  -0.020335979759693146,
  0.08484113961458206,
  -0.4291923940181732,
  0.1903148591518402,
  0.1188591718673706,
  0.09445711970329285,
  0.23127831518650055,
  0.09134680032730103,
  0.14198336005210876,
  -0.06811947375535965,
  0.08434077352285385,
  0.23592601716518402,
  -0.38728341460227966,
  -0.1988343894481659,
  -0.1579573005437851,
  0.41902050375938416,
  0.2773379683494568,
  -0.24203810095787048,
  -0.14796972274780273,
  -0.24934719502925873,
  -2.1141481399536133,
  0.16090790927410126,
  -0.31481626629829407,
  0.3676012456417084,
  -0.05600013583898544,
  -0.23160044848918915,
  -0.3240630030632019,
  -0.21763774752616882,
  0.2679225206375122,
  -0.09008660912513733,
  0.38108891248703003,
  0.07934442162513733,
  0.07929347455501556,
  -0.043119966983795166,
  -0.640408456325531,
  -0.616271436214447,
  -0.44940122961997986,
  -0.5372345447540283,
  -0.055833153426647186,
  0.1766045093536377,
  -0.0805581659078598,
  0.23947353661060333,
  -0.2926062345504761,
  0.20445004105567932,
  0.2890293598175049,
  0.19169005751609802,
  0.2461811751127243,
  0.45519357919692993,
  0.05300622805953026,
  0.08338440954685211,
  0.059107132256031036,
  0.1424563080072403,
  0.10359648615121841,
  -1.8125991821289062,
  0.6122289896011353,
  0.21131697297096252,
  0.29933828115463257,
  -0.30309200286865234,
  -0.2050151228904724,
  0.17677024006843567,
  0.2519862651824951,
  -0.22530046105384827,
  -0.012219816446304321,
  0.0550149567425251,
  0.4090350866317749,
  0.34467020630836487,
  -0.17941823601722717,
  0.07965225726366043,
  -0.19462013244628906,
  -0.0730573907494545,
  0.15700268745422363,
  -0.08526361733675003,
  -0.2224675416946411,
  -0.18683363497257233,
  -0.16962990164756775,
  -0.20338940620422363,
  -0.37420961260795593,
  0.09884975850582123,
  -0.431268572807312,
  0.3009341359138489,
  0.024946924299001694,
  -0.15605495870113373,
  0.07133540511131287,
  -0.033113136887550354,
  0.33995920419692993,
  -0.8444862365722656,
  -0.22354477643966675,
  0.2677299678325653,
  -0.007474236190319061,
  -0.2953784465789795,
  6.364828109741211,
  -0.26100802421569824,
  0.31567585468292236,
  0.19152891635894775,
  -0.19858604669570923,
  0.0659167617559433,
  -0.046197421848773956,
  -0.48397964239120483,
  -0.2746504843235016,
  -0.13124316930770874,
  0.07430057972669601,
  -0.08530586957931519,
  -0.3002980351448059,
  0.3890254497528076,
  -0.5464034080505371,
  0.3074011504650116,
  0.1994432955980301,
  0.5299429893493652,
  0.1389579176902771,
  -0.1955869197845459,
  0.08665124326944351,
  -0.08518073707818985,
  -0.1438159942626953,
  0.39004817605018616,
  -0.29891300201416016,
  0.4396825134754181,
  -0.9011900424957275,
  0.3437999188899994,
  0.6747341752052307,
  0.2589998245239258,
  0.3820466697216034,
  0.47538283467292786,
  -0.38556969165802,
  -0.601749062538147,
  -0.08805722743272781,
  -0.013492906466126442,
  0.05922561511397362,
  0.23348070681095123,
  -0.07214820384979248,
  0.19750814139842987,
  -0.3198971152305603,
  -0.17672164738178253,
  -1.0013163089752197,
  -0.13285978138446808,
  -0.711970329284668,
  -0.23800437152385712,
  0.6168553829193115,
  -0.14332228899002075,
  0.30257803201675415,
  -0.2194240391254425,
  -0.005256421864032745,
  0.02638118714094162,
  0.16391128301620483,
  -0.02454693615436554,
  -0.31334665417671204,
  0.22116418182849884,
  0.1464148610830307,
  0.2139235883951187,
  0.04899539053440094,
  -0.36924776434898376,
  0.11041323840618134,
  0.043286219239234924,
  -0.5844001173973083,
  -0.20701676607131958,
  0.5178765058517456,
  0.13989031314849854,
  -0.7120683193206787,
  -0.1659422665834427,
  0.1841827780008316,
  -0.12600240111351013,
  -0.31906092166900635,
  0.12771159410476685,
  -0.07688009738922119,
  -0.35474538803100586,
  0.24485896527767181,
  0.44496503472328186,
  -0.04793062061071396,
  -0.42645180225372314,
  -0.006770005449652672,
  -0.07747173309326172,
  -0.0497741773724556,
  0.3028721511363983,
  -0.34795963764190674,
  -0.2480904906988144,
  0.2100386768579483,
  0.25284215807914734,
  -0.36693504452705383,
  -0.1384069323539734,
  -0.3103930354118347,
  0.15989452600479126,
  0.20461870729923248,
  -0.17006464302539825,
  0.2410220354795456,
  -0.15801194310188293,
  0.06113314628601074,
  -0.16672733426094055,
  -0.16407343745231628,
  -0.3243131935596466,
  -0.05073188617825508,
  0.038391828536987305,
  -0.37880855798721313,
  0.29700326919555664,
  0.0845111683011055,
  0.04309966415166855,
  0.2742270231246948,
  0.1339433491230011,
  0.2716079652309418,
  -0.2769085168838501,
  -0.12063322216272354,
  0.22837461531162262,
  0.19600118696689606,
  -0.3563724756240845,
  0.08675315976142883,
  0.38844457268714905,
  0.11469197273254395,
  -0.09996815025806427,
  -0.14860771596431732,
  0.018380381166934967,
  0.34475550055503845,
  0.19964924454689026,
  0.37713000178337097,
  0.20551320910453796,
  -0.3779209554195404,
  -0.48662981390953064,
  -1.9429891109466553,
  -0.051101960241794586,
  0.014569271355867386,
  -0.0806940495967865,
  0.19648760557174683,
  -0.41573062539100647,
  0.16220244765281677,
  -0.27358758449554443,
  0.34436464309692383,
  0.35008692741394043,
  0.5995057225227356,
  0.07385148853063583,
  -0.3943208158016205,
  0.19918014109134674,
  0.018997270613908768,
  0.4635632634162903,
  0.11730529367923737,
  0.3416972756385803,
  -0.11983442306518555,
  0.08651156723499298,
  -0.07451950013637543,
  0.046565160155296326,
  -0.2852895259857178,
  -0.29609954357147217,
  0.5215444564819336,
  -0.14081092178821564,
  1.7178703546524048,
  0.39302533864974976,
  0.1303321123123169,
  -0.2530924677848816,
  0.22483472526073456,
  0.06460584700107574,
  0.019857890903949738,
  -0.7595470547676086,
  0.2049836814403534,
  0.08202160149812698,
  0.25852644443511963,
  -0.25044625997543335,
  -0.1990601122379303,
  -0.33349525928497314,
  -0.3074249029159546,
  0.47587987780570984,
  -0.2181563377380371,
  -0.4562060832977295,
  -0.14908066391944885,
  -0.3632863461971283,
  -0.3158543109893799,
  0.2959866523742676,
  -0.49210864305496216,
  0.11091192811727524,
  0.0491027757525444,
  -0.11547216027975082,
  0.3639095723628998,
  0.044658202677965164,
  0.1871337592601776,
  -0.4038384258747101,
  -0.6423047780990601,
  0.13702192902565002,
  -0.507413387298584,
  0.11262470483779907,
  -0.2672097682952881,
  -0.22448965907096863,
  0.3975781202316284,
  -0.48258012533187866,
  0.4123935401439667,
  -0.014700744301080704,
  -0.014367325231432915,
  -0.05274994671344757,
  0.14734166860580444,
  -0.09435930848121643,
  0.10123047232627869,
  0.6356322169303894,
  -0.08682607114315033,
  -0.20362499356269836,
  0.26280689239501953,
  0.049384213984012604,
  0.3331340253353119,
  -0.1461319774389267,
  -0.09449277818202972,
  -0.23187187314033508,
  0.310053288936615,
  -0.19460059702396393,
  0.2412310242652893,
  0.15359725058078766,
  0.19989889860153198,
  0.02930125594139099,
  0.5036065578460693,
  0.088319793343544,
  0.19149266183376312,
  -0.4608885645866394,
  -0.1837993711233139,
  -0.010931704193353653,
  -0.17890097200870514,
  -0.35648083686828613,
  0.31307920813560486,
  0.19638219475746155,
  -2.925781488418579,
  0.43355903029441833,
  0.051135435700416565,
  0.23207442462444305,
  -0.2929818034172058,
  -0.028341419994831085,
  0.0822887271642685,
  0.07871951162815094,
  -0.3724792003631592,
  -0.13542351126670837,
  0.10806901007890701,
  0.323469877243042,
  0.3511023223400116,
  -0.07901158183813095,
  0.014667589217424393,
  0.29353201389312744,
  0.5299474000930786,
  -0.4479430019855499,
  0.30200791358947754,
  0.0028791576623916626,
  0.21829357743263245,
  0.2691461443901062,
  1.8315609693527222,
  -0.28749245405197144,
  0.2809913158416748,
  0.290732204914093,
  -0.004370525479316711,
  0.3379581570625305,
  0.04936140403151512,
  -0.1481609344482422,
  0.19994032382965088,
  -0.11003203690052032,
  0.6180177330970764,
  -0.11561030149459839,
  0.25412052869796753,
  0.294891893863678,
  -0.26607227325439453,
  0.40401726961135864,
  0.03617619723081589,
  0.06719976663589478,
  -0.0012597888708114624,
  -0.09544266760349274,
  -0.30052199959754944,
  -0.1280241310596466,
  0.5860469937324524,
  -0.26407188177108765,
  -0.1796787679195404,
  -0.3560253381729126,
  0.2569483816623688,
  0.18682155013084412,
  -0.35681214928627014,
  -0.13298088312149048,
  -0.12876951694488525,
  0.08908838033676147,
  0.1901300847530365,
  0.2928592562675476,
  -0.3103490173816681,
  -0.25480592250823975,
  -0.051673732697963715,
  -0.1852463334798813,
  0.10264450311660767,
  -0.34124237298965454,
  -0.07334499061107635,
  -0.04394051432609558,
  0.47336843609809875]]
for query_vector in encoded_queries:
    redis_query = (
        Query("(*)=>[KNN 10 @vector $query_vector AS vector_score]")
        .sort_by("vector_score")
        .return_fields("vector_score", "content")
        .dialect(2)
    )
    
    query_params = {"query_vector": np.array(query_vector).astype(np.float32).tobytes()}
    results = client.ft("idx:maxis_documents").search(redis_query, query_params=query_params)

CPU times: user 776 µs, sys: 0 ns, total: 776 µs
Wall time: 598 µs


In [50]:
results

Result{0 total, docs: []}